# 02 - GPU Telemetry Analysis for Anomaly Detection

This notebook analyzes NVIDIA DCGM GPU telemetry patterns to understand what normal vs anomalous GPU behavior looks like during ML training, and how Isolation Forest detects deviations.

In [ ]:
import sys
sys.path.insert(0, '../src')

import json
import pandas as pd
import numpy as np
from collections import Counter

## Load and Extract GPU Events

In [ ]:
with open('../benchmark/data/mlshield_benchmark_v1.json') as f:
    dataset = json.load(f)

gpu_events = []
for traj in dataset:
    for event in traj['events']:
        details = event.get('details', {})
        if 'DCGM_FI_DEV_GPU_UTIL' in details:
            gpu_events.append({
                'job_id': traj['job_id'],
                'label': traj['label'],
                'step': event.get('step', 0),
                'is_malicious': event.get('is_malicious', False),
                'gpu_util': details.get('DCGM_FI_DEV_GPU_UTIL', 0),
                'mem_util': details.get('DCGM_FI_DEV_MEM_COPY_UTIL', 0),
                'fb_used': details.get('DCGM_FI_DEV_FB_USED', 0),
                'gpu_temp': details.get('DCGM_FI_DEV_GPU_TEMP', 0),
                'power': details.get('DCGM_FI_DEV_POWER_USAGE', 0),
                'enc_util': details.get('DCGM_FI_DEV_ENC_UTIL', 0),
            })

gpu_df = pd.DataFrame(gpu_events)
print(f'Total GPU telemetry events: {len(gpu_df):,}')
print(f'Normal: {(~gpu_df["is_malicious"]).sum():,}')
print(f'Malicious: {gpu_df["is_malicious"].sum():,}')

## Normal GPU Telemetry Profile

In [ ]:
normal_gpu = gpu_df[~gpu_df['is_malicious']]
metrics = ['gpu_util', 'mem_util', 'fb_used', 'gpu_temp']

print('=== Normal GPU Telemetry Statistics ===')
print(normal_gpu[metrics].describe().round(2))

## Compare Normal vs Anomalous GPU Patterns by Attack Type

In [ ]:
print('=== GPU Utilization by Label ===')
label_stats = gpu_df.groupby('label')['gpu_util'].agg(['mean', 'std', 'min', 'max', 'count'])
print(label_stats.round(2))

print()
print('=== Memory Utilization by Label ===')
mem_stats = gpu_df.groupby('label')['mem_util'].agg(['mean', 'std', 'min', 'max'])
print(mem_stats.round(2))

## Isolation Forest Anomaly Detection

In [ ]:
from mlshield.detectors.models.isolation import GPUIsolationForest

# Load pre-trained model
iso_forest = GPUIsolationForest()
iso_forest.load('../benchmark/data/models/isolation_forest.pkl')

print(f'Isolation Forest loaded: {iso_forest.is_fitted}')

# Score all GPU events
scores = []
for _, row in gpu_df.iterrows():
    details = {
        'DCGM_FI_DEV_GPU_UTIL': row['gpu_util'],
        'DCGM_FI_DEV_MEM_COPY_UTIL': row['mem_util'],
        'DCGM_FI_DEV_FB_USED': row['fb_used'],
        'DCGM_FI_DEV_GPU_TEMP': row['gpu_temp'],
        'DCGM_FI_DEV_POWER_USAGE': row['power'],
        'DCGM_FI_DEV_ENC_UTIL': row['enc_util'],
    }
    score = iso_forest.score(details)
    scores.append(score if score is not None else 0.0)

gpu_df['anomaly_score'] = scores

print('\n=== Anomaly Score Distribution ===')
print(f'Normal events - Mean: {gpu_df[~gpu_df["is_malicious"]]["anomaly_score"].mean():.4f}')
print(f'Malicious events - Mean: {gpu_df[gpu_df["is_malicious"]]["anomaly_score"].mean():.4f}')

## Score Distribution by Attack Type

In [ ]:
score_by_label = gpu_df.groupby('label')['anomaly_score'].agg(['mean', 'std', 'max'])
print('=== Isolation Forest Scores by Label ===')
print(score_by_label.round(4))

## Thresholding Analysis

Evaluate detection performance at different anomaly score thresholds.

In [ ]:
thresholds = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
print(f'{"Threshold":>10} {"Precision":>10} {"Recall":>10} {"F1":>10}')
print('-' * 42)

for thresh in thresholds:
    preds = gpu_df['anomaly_score'] >= thresh
    truth = gpu_df['is_malicious']
    tp = (preds & truth).sum()
    fp = (preds & ~truth).sum()
    fn = (~preds & truth).sum()
    prec = tp / max(tp + fp, 1)
    rec = tp / max(tp + fn, 1)
    f1 = 2 * prec * rec / max(prec + rec, 1e-10)
    print(f'{thresh:>10.2f} {prec:>10.4f} {rec:>10.4f} {f1:>10.4f}')

## Summary

1. **Normal GPU telemetry** clusters tightly around expected utilization (80-90% GPU, 60-80% memory)
2. **Attack scenarios** show distinct GPU patterns -- cryptojacking shows high utilization, distillation shows low utilization
3. **Isolation Forest** effectively separates normal from anomalous GPU behavior
4. The Isolation Forest is most useful as a complement to the LSTM in MLShield's Layer 2